# Non-Negative Matrix Factorisation (NMF)

## Overview

NMF decomposes a non-negative matrix X ≈ WH, where both W (weights) and H (components) are constrained to be non-negative. This non-negativity constraint produces **parts-based representations** — components are additive, not subtractive, and often directly interpretable.

**When to use NMF:**
- Count data: species abundance matrices, word counts, occurrence records
- Spectral data: NMF naturally separates additive signal sources
- When interpretable, non-negative components are required

**NMF vs PCA:**
| Property | PCA | NMF |
|---|---|---|
| Sign of components | Positive and negative | Non-negative only |
| Interpretability | Moderate | High (additive parts) |
| Data requirement | Any | Non-negative only |
| Orthogonality | Yes | No |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF, PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler

rng = np.random.default_rng(42)
n_sites, n_species = 120, 15
# Simulate a species-by-site count matrix with 3 latent assemblage types
assemblages = np.array([
    [5,4,3,0,0,0,0,0,0,0,0,0,2,1,0],  # upland assemblage
    [0,0,0,4,5,3,0,0,0,0,0,0,0,1,2],  # lowland assemblage
    [0,0,0,0,0,0,5,4,3,2,0,0,0,0,1],  # riparian assemblage
], dtype=float)
site_weights = rng.dirichlet([2,2,2], size=n_sites)
X_counts = (site_weights @ assemblages).astype(float)
X_counts += rng.poisson(0.5, X_counts.shape)  # add noise
species_names = [f"sp_{i:02d}" for i in range(n_species)]
print(f"Species matrix: {X_counts.shape}")
print(f"Non-negative: {(X_counts >= 0).all()}")

---
## Fitting NMF

In [ ]:
nmf = NMF(n_components=3, init="nndsvda", random_state=42, max_iter=500)
W = nmf.fit_transform(X_counts)  # site weights (n_sites x n_components)
H = nmf.components_              # species loadings (n_components x n_species)
print(f"W (site weights): {W.shape}")
print(f"H (species loadings): {H.shape}")
print(f"Reconstruction error: {nmf.reconstruction_err_:.4f}")
X_recon = W @ H
print(f"Correlation original vs reconstructed: {np.corrcoef(X_counts.ravel(), X_recon.ravel())[0,1]:.4f}")

---
## Interpreting Components as Assemblages

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(13,4))
for k in range(3):
    ax = axes[k]
    loadings = pd.Series(H[k], index=species_names)
    loadings.sort_values(ascending=True).plot(kind="barh", ax=ax,
                                               color="steelblue", alpha=0.8)
    ax.set_title(f"NMF Component {k+1}: Species Loadings")
    ax.set_xlabel("Loading (non-negative)")
plt.suptitle("NMF Components as Ecological Assemblage Types", y=1.01)
plt.tight_layout(); plt.show()
for k in range(3):
    top_sp = pd.Series(H[k], index=species_names).nlargest(3).index.tolist()
    print(f"Component {k+1} dominant species: {top_sp}")

---
## Choosing n_components: Reconstruction Error

In [ ]:
ks = range(1, 9)
errors = []
for k in ks:
    nmf_k = NMF(n_components=k, init="nndsvda", random_state=42, max_iter=500)
    nmf_k.fit(X_counts)
    errors.append(nmf_k.reconstruction_err_)
plt.figure(figsize=(7,4))
plt.plot(ks, errors, "o-", color="steelblue", lw=2)
plt.axvline(3, color="#e74c3c", lw=1.5, linestyle="--", label="True k=3")
plt.xlabel("Number of components"); plt.ylabel("Reconstruction error")
plt.title("NMF: Reconstruction Error vs Components (elbow -> true k)")
plt.legend(); plt.tight_layout(); plt.show()

---
## NMF vs PCA on Count Data

In [ ]:
# PCA requires standardisation; NMF works directly on counts
pca = PCA(n_components=2, random_state=42)
X_sc = StandardScaler().fit_transform(X_counts)
X_pca = pca.fit_transform(X_sc)
nmf2 = NMF(n_components=2, init="nndsvda", random_state=42)
W2 = nmf2.fit_transform(X_counts)
# Colour by dominant assemblage type (known ground truth)
true_type = np.argmax(site_weights, axis=1)
fig, axes = plt.subplots(1,2,figsize=(12,5))
for ax, emb, title in zip(axes, [X_pca, W2], ["PCA","NMF"]):
    for k in range(3):
        mask = true_type == k
        ax.scatter(emb[mask,0], emb[mask,1], s=25, alpha=0.7,
                   label=f"Assemblage {k+1}")
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()
print("NMF: each axis represents one assemblage type (interpretable)")
print("PCA: axes are linear combinations with positive and negative loadings")

---

## Common Pitfalls

**1. Applying NMF to data with negative values or after mean-centring**  
NMF strictly requires non-negative input. StandardScaler produces negative values. Use `MinMaxScaler` or work with raw counts. Applying NMF to scaled data with negatives produces nonsensical results.

**2. Using random initialisation (`init="random"`) for reproducibility**  
NMF with random initialisation is sensitive to starting conditions and may converge to different local minima. Use `init="nndsvda"` (non-negative double SVD) for deterministic, better-quality initialisation.

**3. Interpreting NMF components as independent sources**  
NMF components are not orthogonal. A site can have high weight on multiple components simultaneously. Do not treat components as mutually exclusive categories.

**4. Choosing n_components without examining reconstruction error**  
Like k in k-means, the number of NMF components must be selected. Always plot reconstruction error vs n_components and look for the elbow. Cross-validate by holding out a random subset and computing held-out reconstruction error.

**5. Not normalising rows before NMF on abundance data**  
If sites vary greatly in total abundance (e.g. one site with 1000 observations and another with 10), the high-abundance site dominates the decomposition. Row-normalise (divide each row by its sum) before NMF to analyse relative composition.
---
*python_methods_library - Samantha McGarrigle*